# Fourier Feature-Based PINN for TB Parameter Estimation

**Notebook purpose.** This notebook implements an inverse Physics-Informed Neural Network (PINN) framework for reconstructing the eight compartments of a tuberculosis (TB) model and estimating its unknown epidemiological parameters. It also compares reconstructed trajectories with a numerical ODE solution.

- **Model states:** \(S,E_s,I_s,T_s,R,E_r,I_r,T_r\)
- **Observed states:** \(I_s,I_r,T_s,T_r\)
- **Observation period:** weekly observations over the available time interval

> **Before running:** place `tb_datamingguan(1).xlsx` in the notebook's working directory. Run the notebook sequentially from the first cell. Training and repeated estimation/uncertainty analyses can be computationally expensive.

## Contents
1. Environment and data preparation
2. PINN architecture and governing equations
3. PINN training and numerical simulation
4. Results and visualization
5. ODE solver comparison
6. Parameter credibility and identifiability analyses
7. Multi-seed PINN robustness analysis

---

## 1. Environment and data preparation

Import the required libraries, set numerical precision and random seeds, and define the model constants and initial conditions.

In [ ]:
# Core scientific and machine-learning libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Use double precision for the PINN computations
tf.keras.backend.set_floatx("float64")
DTYPE = tf.float64

# Reproducibility
np.random.seed(1234)
tf.random.set_seed(1234)

In [ ]:
EXCEL_PATH = "tb_datamingguan(1).xlsx"
SHEET_NAME = 0
N0 = 288_315_899.0
KNOWN = {"Lambda": 77_557.0, "mu": 0.000269, "gamma_s": 1.0/26.0, "gamma_r": 0.0075,
         "delta_s": 0.000439, "delta_r": 0.00365, "omega": 0.00278, "eta": 0.51}
IC_COUNTS = {"S": 288_016_688.0, "Es": 100_000.0, "Is": 75_690.0, "Ts": 70_796.0,
             "R": 50_000.0, "Er": 1_000.0, "Ir": 941.0, "Tr": 784.0}
STATE_NAMES = ["S","Es","Is","Ts","R","Er","Ir","Tr"]
OBS_NAMES = ["Is","Ir","Ts","Tr"]
OBS_STATE_INDEX = [2, 6, 3, 7]
assert sum(IC_COUNTS.values()) == N0

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
req = ["time"] + OBS_NAMES
df = df[req].apply(pd.to_numeric, errors="coerce").dropna().sort_values("time").reset_index(drop=True)
time_values = df["time"].to_numpy(dtype=np.float64)
assert np.allclose(np.diff(time_values), 1.0)
t0, t_end = float(time_values[0]), float(time_values[-1])
T_SPAN = t_end - t0
N_STEPS = int(round(T_SPAN))
x_data = tf.constant(((time_values - t0)/T_SPAN).reshape(-1,1), dtype=DTYPE)
y_obs_np = df[OBS_NAMES].to_numpy(dtype=np.float64) / N0
y0_np = np.array([IC_COUNTS[s] for s in STATE_NAMES]) / N0
print(len(df), "baris"); display(df.head())

In [ ]:
SC_np = y0_np.copy()
for k, j in enumerate(OBS_STATE_INDEX):
    SC_np[j] = max(SC_np[j], y_obs_np[:, k].max())
SC_np[[1,4,5]] *= 5
SC = tf.constant(SC_np.reshape(1,8), dtype=DTYPE)
z0 = tf.constant((y0_np/SC_np).reshape(1,8), dtype=DTYPE)
z_obs = tf.constant(y_obs_np / SC_np[OBS_STATE_INDEX], dtype=DTYPE)

## 2. PINN architecture and governing equations

The network uses Fourier features as inputs and returns eight scaled compartment trajectories. Unknown epidemiological parameters are constrained to their prescribed bounds through a sigmoid transformation.

In [ ]:
class TB_PINN(tf.keras.Model):
    def __init__(self, width=64, n_fourier=8):
        super().__init__()
        self.B = tf.constant(np.arange(1, n_fourier+1)*np.pi, dtype=DTYPE)
        self.hs = [tf.keras.layers.Dense(width, activation="tanh", dtype=DTYPE) for _ in range(4)]
        self.out = tf.keras.layers.Dense(8, dtype=DTYPE)
    def call(self, x):
        h = tf.concat([x, tf.sin(x*self.B), tf.cos(x*self.B)], axis=1)
        for l in self.hs: h = l(h)
        g = self.out(h)
        return z0 * tf.exp(tf.clip_by_value(x*g, -10., 10.))

model = TB_PINN(); model(tf.zeros((1,1), dtype=DTYPE))

BOUNDS = {"beta_s":(0.05,3.0),"beta_r":(0.05,3.0),"sigma_s":(0.005,0.5),"sigma_r":(0.005,0.5),
          "tau_s":(0.01,1.0),"tau_r":(0.01,1.0),"rho":(0.0,1.0),"kappa":(0.0,1.0)}

class TrainableParameters(tf.Module):
    def __init__(self):
        super().__init__()
        self.raw = {k: tf.Variable(0.0, dtype=DTYPE, name="raw_"+k) for k in BOUNDS}
    def values(self):
        return {k: lo + (hi-lo)*tf.sigmoid(self.raw[k]) for k,(lo,hi) in BOUNDS.items()}
    def variables_list(self):
        return list(self.raw.values())

params = TrainableParameters()

In [ ]:
def rhs_normalized(y, p):
    S, Es, Is, Ts, R, Er, Ir, Tr = [y[:, i:i+1] for i in range(8)]
    N_t = tf.reduce_sum(y, axis=1, keepdims=True)
    lam_s = p["beta_s"]*Is/N_t
    lam_r = p["beta_r"]*Ir/N_t
    c = lambda k: tf.constant(KNOWN[k], dtype=DTYPE)
    Lam = tf.constant(KNOWN["Lambda"]/N0, dtype=DTYPE)
    mu, gs, gr = c("mu"), c("gamma_s"), c("gamma_r")
    ds, dr, om, eta = c("delta_s"), c("delta_r"), c("omega"), c("eta")
    ss, sr, ts, tr = p["sigma_s"], p["sigma_r"], p["tau_s"], p["tau_r"]
    rho, kap = p["rho"], p["kappa"]
    dS  = Lam - (lam_s+lam_r)*S - mu*S + om*R
    dEs = lam_s*(S+eta*R) - (ss+mu)*Es
    dIs = ss*Es + gs*(1-rho)*(1-kap)*Ts - (ts+mu+ds)*Is
    dTs = ts*Is - (gs+mu)*Ts
    dR  = gs*kap*Ts + gr*Tr - eta*(lam_s+lam_r)*R - (om+mu)*R
    dEr = lam_r*(S+eta*R) - (sr+mu)*Er
    dIr = sr*Er + gs*rho*(1-kap)*Ts - (tr+mu+dr)*Ir
    dTr = tr*Ir - (gr+mu)*Tr
    return tf.concat([dS,dEs,dIs,dTs,dR,dEr,dIr,dTr], axis=1)

def rhs_scaled(z, p):
    return rhs_normalized(z*SC, p) / SC

def calculate_residuals(model, params, x):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x)
        z = model(x)
        zs = [z[:, j] for j in range(8)]
    dz = tf.stack([tape.gradient(zj, x)[:, 0] for zj in zs], axis=1) / T_SPAN
    del tape
    return dz - rhs_scaled(z, params.values()), z

def compute_losses(model, params, x_obs, z_obs, x_col):
    res, _ = calculate_residuals(model, params, x_col)
    lp = tf.reduce_mean(tf.square(res))
    zp = tf.gather(model(x_obs), OBS_STATE_INDEX, axis=1)
    ld = tf.reduce_mean(tf.square(zp - z_obs))
    return ld, lp

## 3. PINN training and numerical simulation

The optimization is performed in two stages: data fitting first, followed by joint data-and-physics optimization. The numerical RK4 simulation is subsequently used for trajectory evaluation and parameter refinement.

In [ ]:
history = []
def run(epochs, w_phys_fn, lr, train_vars, print_every=500):
    opt = tf.keras.optimizers.Adam(tf.keras.optimizers.schedules.CosineDecay(lr, epochs))
    for ep in range(epochs):
        x_col = tf.random.uniform((512,1), 0., 1., dtype=DTYPE)
        with tf.GradientTape() as t:
            ld, lp = compute_losses(model, params, x_data, z_obs, x_col)
            loss = 100.0*ld + w_phys_fn(ep, epochs)*lp
        g = t.gradient(loss, train_vars)
        pr = [(a,b) for a,b in zip(g, train_vars) if a is not None]
        cg, _ = tf.clip_by_global_norm([a for a,_ in pr], 5.0)
        opt.apply_gradients(zip(cg, [b for _,b in pr]))
        history.append([float(loss), float(ld), float(lp)])
        if ep % print_every == 0:
            print(ep, f"{float(ld):.3e}", f"{float(lp):.3e}",
                  {k: round(float(v),4) for k,v in params.values().items()})

net_vars = model.trainable_variables
run(5000,  lambda e,n: 0.0, 1e-3, net_vars)
run(15000, lambda e,n: 1e-2 + e/n, 5e-4, net_vars + params.variables_list())

In [ ]:
def simulate(p, n_sub=10):
    dt = 1.0/n_sub; z = z0; out = [z]
    for _ in range(N_STEPS*n_sub):
        k1 = rhs_scaled(z, p);            k2 = rhs_scaled(z+.5*dt*k1, p)
        k3 = rhs_scaled(z+.5*dt*k2, p);   k4 = rhs_scaled(z+dt*k3, p)
        z = z + dt/6*(k1+2*k2+2*k3+k4)
        out.append(z)
    return tf.concat(out[::n_sub], axis=0)

pv = params.variables_list()
opt = tf.keras.optimizers.Adam(1e-2)
for it in range(300):
    with tf.GradientTape() as t:
        zs = tf.gather(simulate(params.values()), OBS_STATE_INDEX, axis=1)
        loss = tf.reduce_mean(tf.square(zs - z_obs))
    opt.apply_gradients(zip(t.gradient(loss, pv), pv))
    if it % 50 == 0: print(it, float(loss))

### 3.1 Saving the trained model

Save network weights, trainable parameter values, scaling constants, and training history for later reuse.

In [ ]:
import h5py, json

os.makedirs("hasil", exist_ok=True)
with h5py.File("hasil/tb_pinn_model.h5", "w") as f:
    g = f.create_group("network_weights")
    for i, w in enumerate(model.get_weights()):
        g.create_dataset(f"w{i:03d}", data=w)

    p = f.create_group("parameters")          # parameter terestimasi (nilai akhir)
    for k, v in params.values().items():
        p.attrs[k] = float(v)
    r = f.create_group("raw_parameters")      # variabel mentah sebelum sigmoid
    for k, v in params.raw.items():
        r.attrs[k] = float(v.numpy())

    s = f.create_group("scaling")
    s.create_dataset("SC", data=SC_np)
    s.create_dataset("z0", data=z0.numpy())
    s.attrs["N0"] = N0
    s.attrs["T_SPAN"] = T_SPAN
    s.attrs["t0"] = t0

    f.attrs["bounds"] = json.dumps(BOUNDS)
    f.attrs["known"] = json.dumps(KNOWN)
    f.attrs["state_names"] = json.dumps(STATE_NAMES)
    f.create_dataset("history", data=np.asarray(history))
print("Tersimpan: hasil/tb_pinn_model.h5")

### 3.2 Reloading the saved model

This section demonstrates how to restore the trained network and parameter values from the HDF5 file.

In [ ]:
model = TB_PINN()
model(tf.zeros((1, 1), dtype=DTYPE))          # membangun layer
params = TrainableParameters()

with h5py.File("hasil/tb_pinn_model.h5", "r") as f:
    g = f["network_weights"]
    model.set_weights([g[k][()] for k in sorted(g.keys())])
    for k, v in f["raw_parameters"].attrs.items():
        params.raw[k].assign(v)

print({k: round(float(v), 4) for k, v in params.values().items()})

## 4. Results and visualization

Inspect the training history, estimated parameters, reconstructed trajectories, residuals, and compartment-level plots.

In [ ]:
h = np.asarray(history)
for j, n in enumerate(["Total","Data","Physics"]):
    plt.semilogy(h[:, j] + 1e-20, label=n)
plt.legend(); plt.grid(True); plt.show()

est = {k: float(v) for k, v in params.values().items()}
display(pd.DataFrame({"Parameter": list(est), "Estimate": list(est.values())}))

sim  = simulate(params.values()).numpy() * SC_np * N0
pinn = model(x_data).numpy() * SC_np * N0
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.plot(time_values, y_obs_np[:, k]*N0, "ko", label="Data")
    ax.plot(time_values, pinn[:, j], "b-", label="PINN")
    ax.plot(time_values, sim[:, j], "r--", label="ODE RK4")
    ax.set_title(OBS_NAMES[k]); ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()

for k, n in enumerate(OBS_NAMES):
    y = y_obs_np[:, k]*N0
    print(n, "RMSE=%.1f" % np.sqrt(np.mean((sim[:, OBS_STATE_INDEX[k]] - y)**2)))

### Training and reconstruction results

In [ ]:
import os
os.makedirs("hasil", exist_ok=True)
cols = ["tab:blue", "tab:red", "tab:green", "tab:orange"]
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.plot(time_values, y_obs_np[:, k]*N0, "ko", ms=4, label="Data aktual")
    ax.plot(time_values, sim[:, j], "-", c=cols[k], lw=2, label="Model ODE")
    ax.plot(time_values, pinn[:, j], "--", c="gray", lw=1.5, label="PINN")
    ax.set_title(OBS_NAMES[k]); ax.set_xlabel("Minggu"); ax.set_ylabel("Jumlah individu")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig("hasil/fit_data_vs_model.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True)

for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    aktual = y_obs_np[:, k] * N0
    prediksi_pinn = pinn[:, j]

    ax.scatter(
        time_values, aktual,
        color="black", s=35, marker="o", zorder=3, label="Actual Data"
    )
    ax.plot(
        time_values, prediksi_pinn,
        color="tab:blue", linewidth=2.5, marker=".", markersize=5,
        label="PINN"
    )

    batas_atas = max(np.max(aktual), np.max(prediksi_pinn))
    ax.set_ylim(0, batas_atas * 1.10 if batas_atas > 0 else 1)

    ax.set_title(OBS_NAMES[k])
    ax.set_xlabel("Week")
    ax.set_ylabel("The number of Individu")
    ax.grid(alpha=0.3)
    ax.legend()

#plt.suptitle("Perbandingan Data Aktual dan Prediksi PINN", fontsize=15)
plt.tight_layout()
plt.savefig("hasil/data_vs_pinn_jelas.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

metode = [
    ("PINN",               pinn, dict(c="tab:blue",  ls="--", lw=2.0)),
    #("ODE RK4 (TF)",       sim,  dict(c="tab:green", ls="-.", lw=2.0)),
    ("ODE solver", ode,  dict(c="tab:red",   ls="-",  lw=2.0)),
]
if "nls" in globals():
    metode.append(("NLS", nls, dict(c="tab:purple", ls=":", lw=2.2)))

eps = 1.0  # menghindari log(0)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    aktual = y_obs_np[:, k] * N0
    ax.scatter(time_values, np.maximum(aktual, eps), color="black", s=30,
               zorder=5, label="Actual Data")
    for nama, M, sty in metode:
        ax.plot(time_values, np.maximum(M[:, j], eps), label=nama, **sty)
    ax.set_yscale("log")
    ax.set_title(OBS_NAMES[k], fontsize=13, fontweight="bold")
    ax.set_ylabel("The Number of Individu")
    ax.grid(alpha=0.3, which="both")

for ax in axs[1]:
    ax.set_xlabel("Week")

h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=11, frameon=False)
#fig.suptitle("Data aktual vs PINN dan ODE (skala log)", fontsize=15)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.savefig("hasil/perbandingan_log.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

metode = [
    ("PINN",               pinn, dict(c="tab:blue", ls="--", lw=2.0)),
    ("ODE solver (SciPy)", ode,  dict(c="tab:red",  ls="-",  lw=2.0)),
]
if "nls" in globals():
    metode.append(("NLS klasik", nls, dict(c="tab:purple", ls=":", lw=2.2)))

eps = 1.0  # menghindari log(0)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    aktual = y_obs_np[:, k] * N0
    ax.scatter(time_values, np.maximum(aktual, eps), color="black", s=30,
               zorder=5, label="Data aktual")
    for nama, M, sty in metode:
        ax.plot(time_values, np.maximum(M[:, j], eps), label=nama, **sty)
    ax.set_yscale("log")
    ax.set_title(OBS_NAMES[k], fontsize=13, fontweight="bold")
    ax.set_ylabel("Jumlah individu (skala log)")
    ax.grid(alpha=0.3, which="both")

for ax in axs[1]:
    ax.set_xlabel("Minggu")

h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=11, frameon=False)
#fig.suptitle("Data aktual vs PINN dan ODE (skala log)", fontsize=15)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.savefig("hasil/perbandingan_log_tanpa_rk4.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    aktual = y_obs_np[:, k] * N0
    ax.scatter(time_values, aktual, c="black", s=30, zorder=5, label="Data aktual")
    ax.plot(time_values, pinn[:, j], "b--", lw=2, label="PINN")
    ax.set_ylabel("Data / PINN", color="tab:blue")
    ax.set_ylim(0, max(aktual.max(), pinn[:, j].max()) * 1.1)

    ax2 = ax.twinx()
    ax2.plot(time_values, ode[:, j], "r-", lw=2, label="ODE (SciPy)")
    ax2.set_ylabel("ODE", color="tab:red")
    ax2.set_ylim(0, ode[:, j].max() * 1.1)

    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)
    ax.set_title(OBS_NAMES[k]); ax.grid(alpha=.3)
for ax in axs[1]: ax.set_xlabel("Minggu")
plt.tight_layout()
plt.savefig("hasil/perbandingan_twin_axis.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
nm = lambda v: v / max(v.max(), 1e-12)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.scatter(time_values, nm(y_obs_np[:, k]*N0), c="black", s=30, zorder=5, label="Data aktual")
    ax.plot(time_values, nm(pinn[:, j]), "b--", lw=2, label="PINN")
    ax.plot(time_values, nm(sim[:, j]), "g-.", lw=2, label="ODE RK4 (TF)")
    ax.plot(time_values, nm(ode[:, j]), "r-", lw=2, label="ODE (SciPy)")
    ax.set_title(OBS_NAMES[k]); ax.set_ylabel("Nilai ternormalisasi (maks = 1)")
    ax.grid(alpha=.3); ax.legend(fontsize=8)
for ax in axs[1]: ax.set_xlabel("Minggu")
plt.tight_layout()
plt.savefig("hasil/perbandingan_normalisasi.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
for j, ax in enumerate(axs.flat):
    ax.plot(time_values, sim[:, j], lw=2)
    if j in OBS_STATE_INDEX:
        k = OBS_STATE_INDEX.index(j)
        ax.plot(time_values, y_obs_np[:, k]*N0, "ko", ms=3)
    ax.set_title(STATE_NAMES[j]); ax.set_xlabel("Minggu"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("hasil/semua_kompartemen.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 8))
for k, ax in enumerate(axs.flat):
    res = sim[:, OBS_STATE_INDEX[k]] - y_obs_np[:, k]*N0
    ax.bar(time_values, res, color=cols[k])
    ax.axhline(0, c="k", lw=.8)
    ax.set_title(f"Residual {OBS_NAMES[k]}"); ax.set_xlabel("Week"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("hasil/residual.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 9))
for k, ax in enumerate(axs.flat):
    y = y_obs_np[:, k]*N0; p_ = sim[:, OBS_STATE_INDEX[k]]
    r2 = 1 - np.sum((y-p_)**2)/np.sum((y-y.mean())**2)
    ax.scatter(y, p_, c=cols[k], s=18)
    lim = [min(y.min(), p_.min()), max(y.max(), p_.max())]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_title(f"{OBS_NAMES[k]}  (R² = {r2:.3f})")
    ax.set_xlabel("Actual"); ax.set_ylabel("Prediction"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("hasil/obs_vs_pred.png", dpi=300); plt.show()

In [ ]:
h = np.asarray(history)
plt.figure(figsize=(10, 5))
for j, n in enumerate(["Total", "Data", "Physics"]):
    plt.semilogy(h[:, j] + 1e-20, label=n)
plt.axvline(5000, c="k", ls=":", label="Step 2")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("hasil/loss_history.png", dpi=300); plt.show()

In [ ]:
names = list(est); vals = [est[n] for n in names]
lo = [BOUNDS[n][0] for n in names]; hi = [BOUNDS[n][1] for n in names]
norm = [(v-l)/(h_-l) for v, l, h_ in zip(vals, lo, hi)]
plt.figure(figsize=(9, 5))
bars = plt.barh(names, norm, color="tab:purple")
for b, v in zip(bars, vals):
    plt.text(b.get_width()+0.01, b.get_y()+b.get_height()/2, f"{v:.4g}", va="center")
plt.xlim(0, 1.2); 
#plt.xlabel("Posisi dalam rentang batas (0 = batas bawah, 1 = batas atas)")
#plt.title("Estimasi parameter"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("hasil/parameter.png", dpi=300); plt.show()

### 4.1 Forward projection beyond the observation period

The following cells extend the trained model trajectory beyond the fitted observation interval. Interpret these values as model-based projections, not as observed data.

In [ ]:
def simulate_n(p, steps, n_sub=10):
    dt = 1.0/n_sub; z = z0; out = [z]
    for _ in range(steps*n_sub):
        k1 = rhs_scaled(z, p);          k2 = rhs_scaled(z+.5*dt*k1, p)
        k3 = rhs_scaled(z+.5*dt*k2, p); k4 = rhs_scaled(z+dt*k3, p)
        z = z + dt/6*(k1+2*k2+2*k3+k4); out.append(z)
    return tf.concat(out[::n_sub], axis=0)

H = N_STEPS + 26
proj = simulate_n(params.values(), H).numpy() * SC_np * N0
tt = np.arange(H+1) + t0
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
for k, ax in enumerate(axs.flat):
    ax.plot(tt, proj[:, OBS_STATE_INDEX[k]], c=cols[k], lw=2, label="Model")
    ax.plot(time_values, y_obs_np[:, k]*N0, "ko", ms=3, label="Data")
    ax.axvline(t_end, c="k", ls=":")
    ax.set_title(OBS_NAMES[k]); ax.set_xlabel("Week"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig("hasil/proyeksi.png", dpi=300); plt.show()

In [ ]:
pd.DataFrame({"Parameter": names, "Estimate": vals}).to_excel("hasil/parameter_estimasi.xlsx", index=False)
out = pd.DataFrame({"time": time_values})
for k, n in enumerate(OBS_NAMES):
    out[n+"_data"] = y_obs_np[:, k]*N0
    out[n+"_model"] = sim[:, OBS_STATE_INDEX[k]]
out.to_excel("hasil/prediksi_vs_data.xlsx", index=False)

### 4.2 Reusing the saved model

Reload the saved model and compare its reconstructed trajectories with the observations and the corresponding ODE simulation.

In [ ]:
import h5py

with h5py.File("hasil/tb_pinn_model.h5", "r") as f:
    SC_np = f["scaling/SC"][()]
    z0 = tf.constant(f["scaling/z0"][()], dtype=DTYPE)   # dipakai oleh TB_PINN.call
    N0_h5 = f["scaling"].attrs["N0"]
    model = TB_PINN()
    model(tf.zeros((1, 1), dtype=DTYPE))
    g = f["network_weights"]
    model.set_weights([g[k][()] for k in sorted(g.keys())])
    params = TrainableParameters()
    for k, v in f["raw_parameters"].attrs.items():
        params.raw[k].assign(v)

pinn_h5 = model(x_data).numpy() * SC_np * N0_h5          # hanya x di [0, 1]

fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    aktual = y_obs_np[:, k] * N0
    ax.scatter(time_values, aktual, c="black", s=30, zorder=5, label="Actual Data")
    ax.plot(time_values, pinn_h5[:, j], c="tab:blue", lw=2.5, marker=".", label="PINN")
    ax.set_ylim(0, max(aktual.max(), pinn_h5[:, j].max()) * 1.1)
    rmse = np.sqrt(np.mean((pinn_h5[:, j] - aktual) ** 2))
    ax.set_title(f"{OBS_NAMES[k]}  (RMSE = {rmse:,.0f})")
    ax.set_xlabel("Week"); ax.set_ylabel("The Number of Individu")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout()
plt.savefig("hasil/prediksi_pinn_h5.png", dpi=300); plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
sim_h5 = simulate_n(params.values(), N_STEPS).numpy() * SC_np * N0_h5
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.scatter(time_values, y_obs_np[:, k]*N0, c="black", s=25, zorder=5, label="Data aktual")
    ax.plot(time_values, pinn_h5[:, j], "b--", lw=2, label="PINN (jaringan)")
    ax.plot(time_values, sim_h5[:, j], "r-", lw=2, label="ODE (parameter PINN)")
    ax.set_yscale("log"); ax.set_title(OBS_NAMES[k]); ax.grid(alpha=.3, which="both"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("hasil/pinn_vs_ode_h5.png", dpi=300); plt.show()

In [ ]:
wk = np.arange(53, 59)                          # minggu 53 sampai 58
x_new = tf.constant(((wk - t0)/T_SPAN).reshape(-1, 1), dtype=DTYPE)
pinn_new = model(x_new).numpy() * SC_np * N0_h5

# Pastikan simulate_n sudah didefinisikan (sel proyeksi sebelumnya)
ode_all = simulate_n(params.values(), int(wk[-1] - t0)).numpy() * SC_np * N0_h5
ode_new = ode_all[(wk - t0).astype(int)]

tabel = pd.DataFrame({"week": wk})
for k, n in enumerate(OBS_NAMES):
    j = OBS_STATE_INDEX[k]
    tabel[n + "_PINN"] = pinn_new[:, j].round(0)
    tabel[n + "_ODE"] = ode_new[:, j].round(0)
display(tabel)

In [ ]:
os.makedirs("hasil", exist_ok=True)

# Excel
tabel.to_excel("hasil/prediksi_minggu53_dst.xlsx", index=False)

# CSV (opsional)
tabel.to_csv("hasil/prediksi_minggu53_dst.csv", index=False)

print("Tersimpan di folder hasil/")

## 5. ODE solver comparison

The SciPy `solve_ivp` implementation integrates the compartmental ODE system using the estimated PINN parameters. The resulting trajectories are evaluated at the observation times.

## 5. ODE solver comparison

In [ ]:
from scipy.integrate import solve_ivp

P = {k: float(v) for k, v in params.values().items()}   # parameter terestimasi

def rhs_np(t, y, P):
    S, Es, Is, Ts, R, Er, Ir, Tr = y
    N = y.sum()
    ls, lr = P["beta_s"]*Is/N, P["beta_r"]*Ir/N
    K = KNOWN
    return [
        K["Lambda"] - (ls+lr)*S - K["mu"]*S + K["omega"]*R,
        ls*(S+K["eta"]*R) - (P["sigma_s"]+K["mu"])*Es,
        P["sigma_s"]*Es + K["gamma_s"]*(1-P["rho"])*(1-P["kappa"])*Ts - (P["tau_s"]+K["mu"]+K["delta_s"])*Is,
        P["tau_s"]*Is - (K["gamma_s"]+K["mu"])*Ts,
        K["gamma_s"]*P["kappa"]*Ts + K["gamma_r"]*Tr - K["eta"]*(ls+lr)*R - (K["omega"]+K["mu"])*R,
        lr*(S+K["eta"]*R) - (P["sigma_r"]+K["mu"])*Er,
        P["sigma_r"]*Er + K["gamma_s"]*P["rho"]*(1-P["kappa"])*Ts - (P["tau_r"]+K["mu"]+K["delta_r"])*Ir,
        P["tau_r"]*Ir - (K["gamma_r"]+K["mu"])*Tr,
    ]

y0_cnt = np.array([IC_COUNTS[s] for s in STATE_NAMES])
sol = solve_ivp(rhs_np, (t0, t_end), y0_cnt, args=(P,), t_eval=time_values,
                method="LSODA", rtol=1e-8, atol=1e-6)
assert sol.success, sol.message
ode = sol.y.T                       # (n_waktu, 8) dalam jumlah individu

def metrics(y, p):
    rmse = np.sqrt(np.mean((p-y)**2))
    mae = np.mean(np.abs(p-y))
    mape = np.mean(np.abs(p-y)/np.maximum(np.abs(y), 1))*100
    r2 = 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2)
    nrmse = rmse/(y.max()-y.min())
    return rmse, mae, mape, r2, nrmse

rows = []
for k, n in enumerate(OBS_NAMES):
    j = OBS_STATE_INDEX[k]
    y = y_obs_np[:, k]*N0
    for nama, pred in [("PINN", pinn[:, j]), ("ODE RK4 (TF)", sim[:, j]), ("ODE solver (SciPy)", ode[:, j])]:
        rmse, mae, mape, r2, nr = metrics(y, pred)
        rows.append([n, nama, rmse, mae, mape, r2, nr])
tabel = pd.DataFrame(rows, columns=["Kompartemen","Metode","RMSE","MAE","MAPE (%)","R2","NRMSE"])
display(tabel.round(4))

# RMSE gabungan, dinormalisasi per kompartemen agar adil antar skala
agg = []
for nama, M in [("PINN", pinn), ("ODE RK4 (TF)", sim), ("ODE solver (SciPy)", ode)]:
    e = [np.sqrt(np.mean((M[:, OBS_STATE_INDEX[k]] - y_obs_np[:, k]*N0)**2)) /
         (y_obs_np[:, k]*N0).mean() for k in range(4)]
    agg.append([nama, np.mean(e)])
display(pd.DataFrame(agg, columns=["Metode","Rata-rata RMSE/mean data"]).round(4))
tabel.to_excel("hasil/perbandingan_metrik.xlsx", index=False)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(13, 9))
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.plot(time_values, y_obs_np[:, k]*N0, "ko", ms=4, label="Data aktual")
    ax.plot(time_values, pinn[:, j], "b--", label="PINN")
    ax.plot(time_values, sim[:, j], "g-.", label="ODE RK4 (TF)")
    ax.plot(time_values, ode[:, j], "r-", lw=2, label="ODE solver (SciPy)")
    ax.set_title(OBS_NAMES[k]); ax.set_xlabel("Minggu"); ax.set_ylabel("Jumlah individu")
    ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig("hasil/perbandingan_prediksi.png", dpi=300); plt.show()

# Bar chart RMSE per kompartemen
piv = tabel.pivot(index="Kompartemen", columns="Metode", values="RMSE").loc[OBS_NAMES]
piv.plot(kind="bar", figsize=(9, 5), logy=True)
plt.ylabel("RMSE (log)"); plt.xticks(rotation=0); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig("hasil/perbandingan_rmse.png", dpi=300); plt.show()

### 5.1 Classical nonlinear least-squares calibration (additional analysis)

This optional section estimates parameters directly by fitting the ODE solution to the observed compartments using nonlinear least squares. It is kept separate from the PINN-based estimates.

In [ ]:
from scipy.optimize import least_squares

keys = list(BOUNDS)
lb = [BOUNDS[k][0] for k in keys]; ub = [BOUNDS[k][1] for k in keys]
obs_cnt = y_obs_np*N0
scale = obs_cnt.mean(axis=0)

def resid(theta):
    Pn = dict(zip(keys, theta))
    s = solve_ivp(rhs_np, (t0, t_end), y0_cnt, args=(Pn,), t_eval=time_values,
                  method="LSODA", rtol=1e-6, atol=1e-6)
    if not s.success or s.y.shape[1] != len(time_values):
        return np.full(obs_cnt.size, 1e6)
    return ((s.y.T[:, OBS_STATE_INDEX] - obs_cnt)/scale).ravel()

x0 = np.array([P[k] for k in keys])          # mulai dari hasil PINN
fit = least_squares(resid, x0, bounds=(lb, ub), x_scale="jac")
P_nls = dict(zip(keys, fit.x))
s = solve_ivp(rhs_np, (t0, t_end), y0_cnt, args=(P_nls,), t_eval=time_values, method="LSODA", rtol=1e-8, atol=1e-6)
nls = s.y.T
for k, n in enumerate(OBS_NAMES):
    print(n, "RMSE NLS = %.1f" % metrics(obs_cnt[:, k], nls[:, OBS_STATE_INDEX[k]])[0])
display(pd.DataFrame({"PINN": P, "NLS": P_nls}).round(5))

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

metode = [
    ("PINN",              pinn, dict(c="tab:blue",   ls="--", lw=1.8)),
    ("ODE RK4 (TF)",      sim,  dict(c="tab:green",  ls="-.", lw=1.8)),
    ("ODE solver (SciPy)", ode, dict(c="tab:red",    ls="-",  lw=2.2)),
]
if "nls" in globals():                       # hanya jika Sel 20 dijalankan
    metode.append(("NLS klasik", nls, dict(c="tab:purple", ls=":", lw=2.2)))

for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    y = y_obs_np[:, k]*N0
    ax.plot(time_values, y, "ko", ms=4, label="Data aktual")
    txt = []
    for nama, M, sty in metode:
        ax.plot(time_values, M[:, j], label=nama, **sty)
        txt.append(f"{nama}: RMSE={np.sqrt(np.mean((M[:, j]-y)**2)):,.0f}")
    ax.set_title(OBS_NAMES[k], fontsize=13, fontweight="bold")
    ax.set_ylabel("Jumlah individu"); ax.grid(alpha=.3)
    ax.text(0.02, 0.98, "\n".join(txt), transform=ax.transAxes, va="top", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", alpha=.85))
for ax in axs[1]: ax.set_xlabel("Minggu")

h, l = axs[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=11, frameon=False)
fig.suptitle("Perbandingan prediksi semua metode terhadap data aktual", fontsize=15)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.savefig("hasil/perbandingan_semua_metode.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Parameter credibility and identifiability analyses

The following supplementary analyses examine the stability of parameter estimates, profile-based intervals, local uncertainty, bootstrap variability, residual diagnostics, and sensitivity to fixed parameter values.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy import stats

keys = list(BOUNDS)
lb = np.array([BOUNDS[k][0] for k in keys]); ub = np.array([BOUNDS[k][1] for k in keys])
obs_cnt = y_obs_np * N0
scale = obs_cnt.mean(axis=0)
n_obs, n_par = obs_cnt.size, len(keys)

def model_obs(theta, tgrid=time_values):
    Pn = dict(zip(keys, theta))
    s = solve_ivp(rhs_np, (t0, tgrid[-1]), y0_cnt, args=(Pn,), t_eval=tgrid,
                  method="LSODA", rtol=1e-7, atol=1e-6)
    if (not s.success) or s.y.shape[1] != len(tgrid):
        return None
    return s.y.T[:, OBS_STATE_INDEX]

def resid(theta, data=None):
    data = obs_cnt if data is None else data
    m = model_obs(theta)
    if m is None:
        return np.full(data.size, 1e3)
    return ((m - data) / scale).ravel()

def fit(theta0, data=None, fixed=None):
    """fixed: dict {indeks_parameter: nilai} untuk profile likelihood."""
    free = [i for i in range(n_par) if not fixed or i not in fixed]
    def f(z):
        th = np.array(theta0, float)
        if fixed:
            for i, v in fixed.items(): th[i] = v
        th[free] = z
        return resid(th, data)
    r = least_squares(f, np.array(theta0)[free], bounds=(lb[free], ub[free]), x_scale="jac")
    th = np.array(theta0, float)
    if fixed:
        for i, v in fixed.items(): th[i] = v
    th[free] = r.x
    return th, 2 * r.cost          # RSS

theta_hat = np.array([P[k] for k in keys])        # mulai dari hasil PINN
theta_hat, rss_min = fit(theta_hat)
print("RSS min:", rss_min); print(dict(zip(keys, theta_hat.round(5))))

In [ ]:
rng = np.random.default_rng(1234)
starts, rows = 20, []
for s in range(starts):
    th0 = lb + (ub - lb) * rng.uniform(0.05, 0.95, n_par)
    try:
        th, rss = fit(th0)
        rows.append([s, rss, *th])
    except Exception as e:
        print("start", s, "gagal:", e)
ms = pd.DataFrame(rows, columns=["start", "RSS", *keys])
ms_good = ms[ms.RSS <= ms.RSS.min() * 1.05]       # yang konvergen ke optimum yang sama
summ = pd.DataFrame({"mean": ms_good[keys].mean(), "SD": ms_good[keys].std(),
                     "CV (%)": ms_good[keys].std() / ms_good[keys].mean() * 100,
                     "min": ms_good[keys].min(), "max": ms_good[keys].max(),
                     "batas_bawah": lb, "batas_atas": ub})
display(ms.sort_values("RSS").round(5)); display(summ.round(5))
print(f"{len(ms_good)}/{len(ms)} start mencapai RSS dalam 5% dari terbaik")

ax = ms_good[keys].boxplot(figsize=(10, 4)); ax.set_yscale("log")
#plt.title("Sebaran parameter multi-start"); plt.tight_layout()
plt.savefig("hasil/multistart_boxplot.png", dpi=300); plt.show()
summ.to_excel("hasil/multistart_ringkasan.xlsx")

In [ ]:
grid_n = 15
alpha_thr = stats.chi2.ppf(0.95, 1)
prof = {}
for i, k in enumerate(keys):
    vals = np.linspace(lb[i], ub[i], grid_n)
    # tambahkan optimum agar kurva melewati minimum
    vals = np.sort(np.append(vals, theta_hat[i]))
    th_cur, out = theta_hat.copy(), []
    for v in vals:
        try:
            th_cur, rss = fit(th_cur, fixed={i: v})
        except Exception:
            rss = np.nan
        out.append(rss)
    prof[k] = (vals, np.array(out))
    print("selesai", k)

fig, axs = plt.subplots(2, 4, figsize=(18, 8))
ci_rows = []
for ax, (i, k) in zip(axs.flat, enumerate(keys)):
    vals, rss = prof[k]
    dev = n_obs * np.log(rss / rss_min)           # deviance berbasis RSS
    ax.plot(vals, dev, "o-")
    ax.axhline(alpha_thr, c="r", ls="--", label="95% (chi²,1)")
    ax.axvline(theta_hat[i], c="k", ls=":")
    ax.set_yscale("symlog", linthresh=1); ax.set_title(k); ax.grid(alpha=.3)
    inside = vals[dev <= alpha_thr]
    lo, hi = (inside.min(), inside.max()) if len(inside) else (np.nan, np.nan)
    flat = (lo <= lb[i] * 1.001) or (hi >= ub[i] * 0.999)
    ci_rows.append([k, theta_hat[i], lo, hi, "TIDAK teridentifikasi (menyentuh batas)" if flat else "teridentifikasi"])
axs.flat[0].legend()
plt.tight_layout(); plt.savefig("hasil/profile_likelihood.png", dpi=300); plt.show()
ci_pl = pd.DataFrame(ci_rows, columns=["Parameter", "Estimasi", "CI95_bawah", "CI95_atas", "Status"])
display(ci_pl.round(5)); ci_pl.to_excel("hasil/profile_likelihood_ci.xlsx", index=False)

In [ ]:
def jac_num(th, h=1e-4):
    base = resid(th); J = np.zeros((base.size, n_par))
    for i in range(n_par):
        d = h * max(abs(th[i]), 1e-3)
        t2 = th.copy(); t2[i] += d
        J[:, i] = (resid(t2) - base) / d
    return J

J = jac_num(theta_hat)
sigma2 = rss_min / (n_obs - n_par)
cov = sigma2 * np.linalg.pinv(J.T @ J)
se = np.sqrt(np.diag(cov))
corr = cov / np.outer(se, se)
sv = np.linalg.svd(J, compute_uv=False)
print("Nilai singular:", sv.round(6))
print("Kondisi (maks/min): %.3e  | rank numerik: %d dari %d" % (sv[0]/sv[-1], np.sum(sv > sv[0]*1e-6), n_par))

fim_tab = pd.DataFrame({"Estimasi": theta_hat, "SE": se, "CI95_bawah": theta_hat - 1.96*se,
                        "CI95_atas": theta_hat + 1.96*se, "RSE (%)": se / theta_hat * 100}, index=keys)
display(fim_tab.round(5))

plt.figure(figsize=(7, 6))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.xticks(range(n_par), keys, rotation=45); plt.yticks(range(n_par), keys); plt.colorbar()
plt.title("Korelasi antar parameter"); plt.tight_layout()
plt.savefig("hasil/korelasi_parameter.png", dpi=300); plt.show()

In [ ]:
n_boot = 100
m_hat = model_obs(theta_hat)
res = obs_cnt - m_hat
boot_th, boot_pred = [], []
for b in range(n_boot):
    ystar = m_hat + res[rng.integers(0, len(res), len(res))]    # resampling per baris waktu
    ystar = np.maximum(ystar, 0)
    try:
        th, _ = fit(theta_hat, data=ystar)
        boot_th.append(th); boot_pred.append(model_obs(th))
    except Exception:
        pass
boot_th = np.array(boot_th); boot_pred = np.array(boot_pred)
print("bootstrap sukses:", len(boot_th), "/", n_boot)

bt = pd.DataFrame({"Estimasi": theta_hat, "Mean_boot": boot_th.mean(0), "SD_boot": boot_th.std(0),
                   "CI95_bawah": np.percentile(boot_th, 2.5, 0), "CI95_atas": np.percentile(boot_th, 97.5, 0)}, index=keys)
display(bt.round(5)); bt.to_excel("hasil/bootstrap_parameter.xlsx")

lo_b, hi_b = np.percentile(boot_pred, [2.5, 97.5], axis=0)
fig, axs = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for k, ax in enumerate(axs.flat):
    ax.fill_between(time_values, lo_b[:, k], hi_b[:, k], color="tab:red", alpha=.25, label="Pita 95% bootstrap")
    ax.plot(time_values, m_hat[:, k], "r-", lw=2, label="ODE terbaik")
    ax.scatter(time_values, obs_cnt[:, k], c="k", s=20, zorder=5, label="Data")
    ax.set_title(OBS_NAMES[k]); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("hasil/pita_kepercayaan.png", dpi=300); plt.show()

In [ ]:
# pip install statsmodels
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

fig, axs = plt.subplots(3, 4, figsize=(18, 10))
diag = []
for k, n in enumerate(OBS_NAMES):
    r = (obs_cnt[:, k] - m_hat[:, k])
    axs[0, k].bar(time_values, r); axs[0, k].axhline(0, c="k", lw=.8); axs[0, k].set_title(f"Residual {n}")
    plot_acf(r, lags=min(20, len(r)//2 - 1), ax=axs[1, k]); axs[1, k].set_title(f"ACF {n}")
    stats.probplot(r, dist="norm", plot=axs[2, k]); axs[2, k].set_title(f"Q-Q {n}")
    sw = stats.shapiro(r).pvalue
    lb_p = acorr_ljungbox(r, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    diag.append([n, np.sqrt(np.mean(r**2)), sw, lb_p,
                 "normal" if sw > .05 else "tidak normal", "tidak ada autokorelasi" if lb_p > .05 else "ada autokorelasi"])
plt.tight_layout(); plt.savefig("hasil/diagnostik_residual.png", dpi=300); plt.show()
diag = pd.DataFrame(diag, columns=["Kompartemen", "RMSE", "p_Shapiro", "p_LjungBox(10)", "Normalitas", "Autokorelasi"])
display(diag.round(4)); diag.to_excel("hasil/diagnostik_residual.xlsx", index=False)

In [ ]:
fixed_names = ["Lambda", "omega", "eta", "mu", "gamma_s", "gamma_r", "delta_s", "delta_r"]
KNOWN_ORI = KNOWN.copy()
rows = []
for nm in fixed_names:
    for f_ in (0.8, 1.2):
        KNOWN.update(KNOWN_ORI); KNOWN[nm] = KNOWN_ORI[nm] * f_
        try:
            th, rss = fit(theta_hat)
            rows.append([nm, f"{int((f_-1)*100):+d}%", rss, *th])
        except Exception:
            pass
KNOWN.update(KNOWN_ORI)                              # pulihkan nilai asli
sens = pd.DataFrame(rows, columns=["Parameter tetap", "Perubahan", "RSS", *keys])
for k in keys:
    sens["Δ% " + k] = (sens[k] - theta_hat[keys.index(k)]) / theta_hat[keys.index(k)] * 100
display(sens.round(4)); sens.to_excel("hasil/sensitivitas_parameter_tetap.xlsx", index=False)

pivot = sens.pivot_table(index="Parameter tetap", values=["Δ% " + k for k in keys], aggfunc=lambda x: np.max(np.abs(x)))
ax = pivot.plot(kind="barh", figsize=(10, 6)); ax.set_xlabel("Perubahan maksimum estimasi (%)")
plt.title("Sensitivitas estimasi terhadap parameter tetap ±20%"); plt.tight_layout()
plt.savefig("hasil/tornado_sensitivitas.png", dpi=300); plt.show()

In [ ]:
ringkas = pd.DataFrame({
    "Estimasi": theta_hat,
    "RSE_Fisher (%)": se / theta_hat * 100,
    "CV_multistart (%)": summ["CV (%)"].values,
    "Status_profile": ci_pl["Status"].values,
    "CI_boot_bawah": bt["CI95_bawah"].values, "CI_boot_atas": bt["CI95_atas"].values,
}, index=keys)
display(ringkas.round(4)); ringkas.to_excel("hasil/ringkasan_identifiabilitas.xlsx")

## 7. Multi-seed PINN robustness analysis

Repeat PINN training with different random seeds to assess variability in the estimated parameters and reconstructed trajectories. The epoch counts below are set to reduced values for experimentation; increase them for final computational experiments if required.

In [ ]:
import time
os.makedirs("hasil", exist_ok=True)
keys = list(BOUNDS)
EP1, EP2, EP3 = 3000, 7000, 150      # naikkan (5000/15000/300) untuk hasil akhir
obs_cnt = y_obs_np * N0

def train_pinn(seed, z_target=None, ep1=EP1, ep2=EP2, ep3=EP3, verbose=False):
    z_target = z_obs if z_target is None else z_target
    tf.random.set_seed(seed); np.random.seed(seed)
    m = TB_PINN(); m(tf.zeros((1, 1), dtype=DTYPE))
    p = TrainableParameters()
    for v in p.raw.values():                       # inisialisasi parameter acak per seed
        v.assign(np.random.uniform(-1.5, 1.5))

    def stage(epochs, wfn, lr, tv):
        opt = tf.keras.optimizers.Adam(tf.keras.optimizers.schedules.CosineDecay(lr, epochs))
        for ep in range(epochs):
            xc = tf.random.uniform((512, 1), 0., 1., dtype=DTYPE)
            with tf.GradientTape() as t:
                ld, lp = compute_losses(m, p, x_data, z_target, xc)
                loss = 100.0*ld + wfn(ep, epochs)*lp
            g = t.gradient(loss, tv)
            pr = [(a, b) for a, b in zip(g, tv) if a is not None]
            cg, _ = tf.clip_by_global_norm([a for a, _ in pr], 5.0)
            opt.apply_gradients(zip(cg, [b for _, b in pr]))
            if verbose and ep % 1000 == 0:
                print(seed, ep, float(ld), float(lp))
        return float(ld), float(lp)

    nv = m.trainable_variables
    stage(ep1, lambda e, n: 0.0, 1e-3, nv)
    ld, lp = stage(ep2, lambda e, n: 1e-2 + e/n, 5e-4, nv + p.variables_list())

    if ep3 > 0:                                    # refinement shooting pada parameter
        pv = p.variables_list(); o3 = tf.keras.optimizers.Adam(1e-2)
        for _ in range(ep3):
            with tf.GradientTape() as t:
                zs = tf.gather(simulate(p.values()), OBS_STATE_INDEX, axis=1)
                l3 = tf.reduce_mean(tf.square(zs - z_target))
            o3.apply_gradients(zip(t.gradient(l3, pv), pv))
    th = {k: float(v) for k, v in p.values().items()}
    zpred = tf.gather(m(x_data), OBS_STATE_INDEX, axis=1).numpy()
    zode = tf.gather(simulate(p.values()), OBS_STATE_INDEX, axis=1).numpy()
    return dict(theta=th, loss_data=ld, loss_phys=lp, pinn=m(x_data).numpy()*SC_np*N0,
                ode=simulate(p.values()).numpy()*SC_np*N0, model=m)

In [ ]:
N_SEED = 10
runs, rows = [], []
for s in range(N_SEED):
    t_ = time.time()
    r = train_pinn(seed=100 + s)
    rmse_ode = np.mean([np.sqrt(np.mean((r["ode"][:, OBS_STATE_INDEX[k]] - obs_cnt[:, k])**2)) for k in range(4)])
    rmse_pinn = np.mean([np.sqrt(np.mean((r["pinn"][:, OBS_STATE_INDEX[k]] - obs_cnt[:, k])**2)) for k in range(4)])
    rows.append(dict(seed=100+s, loss_data=r["loss_data"], loss_phys=r["loss_phys"],
                     RMSE_PINN=rmse_pinn, RMSE_ODE=rmse_ode, **r["theta"]))
    runs.append(r)
    print(f"seed {100+s} selesai ({time.time()-t_:.0f} s) RMSE_ODE={rmse_ode:,.0f}")
seeds_df = pd.DataFrame(rows)
display(seeds_df.round(5)); seeds_df.to_excel("hasil/pinn_multiseed.xlsx", index=False)

In [ ]:
best = seeds_df.RMSE_ODE.idxmin()
good = seeds_df[seeds_df.RMSE_ODE <= seeds_df.RMSE_ODE.min() * 1.5]   # seed yang fit-nya layak
A = good[keys]
summ = pd.DataFrame({
    "mean": A.mean(), "SD": A.std(), "CV (%)": A.std()/A.mean()*100,
    "median": A.median(), "CI95_bawah": A.quantile(.025), "CI95_atas": A.quantile(.975),
    "batas_bawah": [BOUNDS[k][0] for k in keys], "batas_atas": [BOUNDS[k][1] for k in keys],
    "terbaik": seeds_df.loc[best, keys]})
summ["posisi_dalam_batas"] = [(summ.loc[k, "mean"]-BOUNDS[k][0])/(BOUNDS[k][1]-BOUNDS[k][0]) for k in keys]
summ["Status"] = np.where((summ["CV (%)"] < 10) & summ.posisi_dalam_batas.between(.02, .98),
                          "teridentifikasi", np.where(summ["CV (%)"] < 30, "lemah", "tidak teridentifikasi"))
display(summ.round(5)); summ.to_excel("hasil/pinn_ringkasan_parameter.xlsx")
print(f"{len(good)}/{len(seeds_df)} seed dipakai (RMSE ODE <= 1.5 x terbaik)")

ax = A.boxplot(figsize=(10, 4)); ax.set_yscale("log")
plt.title("Sebaran parameter hasil PINN antar seed"); plt.tight_layout()
plt.savefig("hasil/pinn_boxplot_parameter.png", dpi=300); plt.show()

plt.figure(figsize=(7, 6)); C = A.corr().values
plt.imshow(C, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(8), keys, rotation=45); plt.yticks(range(8), keys)
plt.title("Korelasi parameter antar seed PINN"); plt.tight_layout()
plt.savefig("hasil/pinn_korelasi.png", dpi=300); plt.show()

In [ ]:
sel = [runs[i] for i in good.index]
ode_s = np.array([r["ode"] for r in sel]); pinn_s = np.array([r["pinn"] for r in sel])
fig, axs = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for k, ax in enumerate(axs.flat):
    j = OBS_STATE_INDEX[k]
    ax.fill_between(time_values, *np.percentile(ode_s[:, :, j], [2.5, 97.5], axis=0), color="tab:red", alpha=.2, label="Pita 95% ODE (antar seed)")
    ax.plot(time_values, np.median(ode_s[:, :, j], axis=0), "r-", lw=2, label="ODE (median)")
    ax.plot(time_values, np.median(pinn_s[:, :, j], axis=0), "b--", lw=2, label="PINN (median)")
    ax.scatter(time_values, obs_cnt[:, k], c="k", s=22, zorder=5, label="Data")
    ax.set_yscale("log"); ax.set_title(OBS_NAMES[k]); ax.grid(alpha=.3, which="both"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("hasil/pinn_pita_ketidakpastian.png", dpi=300); plt.show()

In [ ]:
N_BOOT = 20                                       # mahal; tiap iterasi = 1 training PINN
best_run = runs[best]
fit_cnt = best_run["pinn"][:, OBS_STATE_INDEX]
res = obs_cnt - fit_cnt
rng = np.random.default_rng(7)
boot = []
for b in range(N_BOOT):
    ystar = np.maximum(fit_cnt + res[rng.integers(0, len(res), len(res))], 0)
    z_t = tf.constant(ystar / N0 / SC_np[OBS_STATE_INDEX], dtype=DTYPE)
    r = train_pinn(seed=500 + b, z_target=z_t)
    boot.append(r["theta"]); print("boot", b + 1, "/", N_BOOT)
boot_df = pd.DataFrame(boot)
bt = pd.DataFrame({"Estimasi_terbaik": summ["terbaik"], "Mean_boot": boot_df.mean(), "SD_boot": boot_df.std(),
                   "CI95_bawah": boot_df.quantile(.025), "CI95_atas": boot_df.quantile(.975)})
display(bt.round(5)); bt.to_excel("hasil/pinn_bootstrap.xlsx")

In [ ]:
# pip install statsmodels
from scipy import stats
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

fig, axs = plt.subplots(3, 4, figsize=(18, 10)); diag = []
for k, n in enumerate(OBS_NAMES):
    r_ode = obs_cnt[:, k] - best_run["ode"][:, OBS_STATE_INDEX[k]]
    axs[0, k].bar(time_values, r_ode); axs[0, k].axhline(0, c="k", lw=.8); axs[0, k].set_title(f"Residual ODE {n}")
    plot_acf(r_ode, lags=20, ax=axs[1, k]); axs[1, k].set_title(f"ACF {n}")
    stats.probplot(r_ode, dist="norm", plot=axs[2, k]); axs[2, k].set_title(f"Q-Q {n}")
    sw = stats.shapiro(r_ode).pvalue
    lbp = acorr_ljungbox(r_ode, lags=[10], return_df=True)["lb_pvalue"].iloc[0]
    diag.append([n, np.sqrt(np.mean(r_ode**2)), np.mean(r_ode), sw, lbp])
plt.tight_layout(); plt.savefig("hasil/pinn_diagnostik_residual.png", dpi=300); plt.show()
diag = pd.DataFrame(diag, columns=["Kompartemen", "RMSE", "Bias(mean residual)", "p_Shapiro", "p_LjungBox(10)"])
display(diag.round(4)); diag.to_excel("hasil/pinn_diagnostik_residual.xlsx", index=False)

In [ ]:
FIX = ["Lambda", "omega", "eta"]                  # tambah nama lain bila perlu (mahal)
KNOWN_ORI = KNOWN.copy(); rows = []
for nm in FIX:
    for f_ in (0.8, 1.2):
        KNOWN.update(KNOWN_ORI); KNOWN[nm] = KNOWN_ORI[nm] * f_
        r = train_pinn(seed=100 + best)           # seed sama agar perbedaan hanya akibat KNOWN
        rows.append(dict(tetap=nm, ubah=f"{int((f_-1)*100):+d}%", **r["theta"]))
        print(nm, f_)
KNOWN.update(KNOWN_ORI)
sens = pd.DataFrame(rows)
for k in keys:
    sens["Δ% " + k] = (sens[k] - summ.loc[k, "terbaik"]) / summ.loc[k, "terbaik"] * 100
display(sens.round(4)); sens.to_excel("hasil/pinn_sensitivitas_tetap.xlsx", index=False)
ax = sens.groupby("tetap")[["Δ% " + k for k in keys]].agg(lambda x: np.max(np.abs(x))).plot(kind="barh", figsize=(10, 5))
ax.set_xlabel("Perubahan maks estimasi (%)"); plt.tight_layout()
plt.savefig("hasil/pinn_tornado.png", dpi=300); plt.show()

In [ ]:
def R0_strain(th, K=KNOWN):
    S0 = K["Lambda"]/K["mu"]; N = S0                       # populasi bebas penyakit (dalam jumlah)
    out = {}
    for s in ("s", "r"):
        sig, tau, dl = th["sigma_"+s], th["tau_"+s], K["delta_"+s]
        gam = K["gamma_"+s]; mu = K["mu"]
        # E -> I -> T; T kembali ke I sebagian (lihat model) → pendekatan tanpa umpan balik
        out["R0_"+s] = th["beta_"+s] * sig/(sig+mu) * 1/(tau+mu+dl)
    return out
R0_df = pd.DataFrame([R0_strain({k: row[k] for k in keys}) for _, row in good.iterrows()])
display(R0_df.describe().loc[["mean", "std", "min", "max"]].round(3))